In [25]:
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field 
from langchain_core.output_parsers import StrOutputParser
import os 
from langchain_community.graphs import Neo4jGraph 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama 
from langchain_experimental.graph_transformers import LLMGraphTransformer 
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget 
from langchain_neo4j import Neo4jVector 
from langchain_openai import OpenAIEmbeddings 
from langchain_community.document_loaders import TextLoader 
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars  
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
graph = Neo4jGraph()

C:\Users\Shervin Zare\AppData\Local\Temp\ipykernel_11052\267675202.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph()


In [3]:
loader = TextLoader("dummytext.txt")
docs = loader.load()

textsplitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=24)
documents = textsplitter.split_documents(docs)

In [14]:
llm = OpenAI(model="gpt-4o-mini", temperature=0)

llm_transformer = LLMGraphTransformer(
    llm=llm,
    additional_instructions="Every extracted node must have a valid, non-empty type or label. Never leave the node type blank."
)

graph_documents = llm_transformer.convert_to_graph_documents(documents)

print("Saved cleanly to Neo4j Aura!")

Saved cleanly to Neo4j Aura!


In [ ]:
## Store it in neo4j 
graph.add_graph_documents(graph_documents, include_source=True, baseEntityLabel=True)

In [17]:
def showGraph():
    driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"])
    )
    session = driver.session() 
    widget = GraphWidget(graph = session.run("Match (s)-[r:!MENTIONS]-> (t) RETURN s,r,t").graph())
    widget.node_labeling_map = 'id'
    return widget 

showGraph()

GraphWidget(layout=Layout(height='800px', width='100%'))

In [18]:
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(), 
    search_type="hybrid", 
    node_label = "Document", 
    text_node_properties=["text"], 
    embedding_node_property="embedding"

)
vector_retriever = vector_index.as_retriever()

In [ ]:
class Entities(BaseModel):
    """Identifying information about entities."""
    names: list[str] = Field(
        ...,
        description="All the person, organization, or business entities that appear in the text",
    )


In [26]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following input: {question}",
        ),
    ]
)

# 3. Model setup & LCEL pipeline
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

entity_chain = prompt | llm.with_structured_output(Entities)

In [29]:
entity_chain.invoke({"question": "Who are Nonna Lucia and Giovani Caruso?"}).names

['Nonna Lucia', 'Giovani Caruso']